<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Automatización de procesos</p>
  </div>

</div>

# **Proyecto de automatización de datos y servicios**
## *API + scraping + procesamiento + configuración + Excel + auditoría*

Este laboratorio cierra el bloque de automatización de datos y servicios.

Integraremos:

```text
API
  +
Web scraping
  ↓
validación
  ↓
normalización
  ↓
integración de datos
  ↓
configuración externa
  ↓
variables de entorno
  ↓
Excel
  ↓
auditoría
  ↓
estructura de proyecto
  ↓
Git/GitHub
```

El objetivo es construir un proyecto reproducible que conecte varias fuentes de datos y genere un informe final automáticamente.

## Objetivos

Al finalizar deberías poder:

1. Diseñar un proyecto de automatización con varias fuentes.
2. Consumir una API.
3. Extraer HTML.
4. Trabajar con fallback local.
5. Validar esquemas.
6. Normalizar fuentes heterogéneas.
7. Integrar datos.
8. Separar configuración del código.
9. Utilizar variables de entorno.
10. Generar un Excel final.
11. Crear auditoría del proceso.
12. Crear estructura profesional de proyecto.
13. Documentar ejecución.
14. Preparar el proyecto para Git/GitHub.

## 1. Preparación

In [ ]:
from pathlib import Path
import json
import os
import shutil
import time
import textwrap

import pandas as pd
import matplotlib.pyplot as plt

try:
    import requests
    REQUESTS_DISPONIBLE = True
except ImportError:
    REQUESTS_DISPONIBLE = False

try:
    from bs4 import BeautifulSoup
    BS4_DISPONIBLE = True
except ImportError:
    BS4_DISPONIBLE = False

try:
    from dotenv import load_dotenv
    DOTENV_DISPONIBLE = True
except ImportError:
    DOTENV_DISPONIBLE = False

try:
    from openpyxl import Workbook, load_workbook
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.chart import BarChart, Reference
    from openpyxl.utils import get_column_letter
    OPENPYXL_DISPONIBLE = True
except ImportError:
    OPENPYXL_DISPONIBLE = False

print("requests:", REQUESTS_DISPONIBLE)
print("bs4:", BS4_DISPONIBLE)
print("dotenv:", DOTENV_DISPONIBLE)
print("openpyxl:", OPENPYXL_DISPONIBLE)

## 2. Crear estructura de proyecto

In [ ]:
raiz = Path("proyecto_automatizacion_servicios")

if raiz.exists():
    shutil.rmtree(raiz)

carpetas = [
    raiz / "data" / "raw",
    raiz / "data" / "processed",
    raiz / "outputs",
    raiz / "logs",
    raiz / "config",
    raiz / "src"
]

for carpeta in carpetas:
    carpeta.mkdir(
        parents=True,
        exist_ok=True
    )

print("Proyecto creado:", raiz.resolve())

## 2.1 Visualizar estructura

In [ ]:
for ruta in sorted(raiz.rglob("*")):
    nivel = len(ruta.relative_to(raiz).parts) - 1
    simbolo = "📁" if ruta.is_dir() else "📄"
    print("    " * nivel + simbolo + " " + ruta.name)

## 3. Configuración externa JSON

In [ ]:
config = {
    "api_url":
        "https://jsonplaceholder.typicode.com/users",

    "timeout":
        10,

    "excel_output":
        "outputs/informe_integrado.xlsx",

    "csv_output":
        "data/processed/datos_integrados.csv"
}

ruta_config = (
    raiz
    / "config"
    / "config.json"
)

ruta_config.write_text(
    json.dumps(
        config,
        indent=2
    ),
    encoding="utf-8"
)

print(
    ruta_config.read_text(
        encoding="utf-8"
    )
)

## 4. Variables de entorno

In [ ]:
env_example = '''
API_TOKEN=
ENVIRONMENT=development
'''

(
    raiz
    / ".env.example"
).write_text(
    textwrap.dedent(
        env_example
    ).strip()
    + "\n",
    encoding="utf-8"
)

(
    raiz
    / ".env"
).write_text(
    "API_TOKEN=token_demo_no_real\n"
    "ENVIRONMENT=development\n",
    encoding="utf-8"
)

print(
    (raiz / ".env.example").read_text(
        encoding="utf-8"
    )
)

## 4.1 Cargar entorno

In [ ]:
if DOTENV_DISPONIBLE:

    load_dotenv(
        raiz / ".env",
        override=True
    )

    print(
        "ENVIRONMENT:",
        os.getenv("ENVIRONMENT")
    )

    print(
        "API_TOKEN configurado:",
        bool(
            os.getenv("API_TOKEN")
        )
    )
else:
    print(
        "Instalar python-dotenv."
    )

## 5. Primera fuente: API

In [ ]:
API_URL = config["api_url"]

datos_api = None
status_api = None

if REQUESTS_DISPONIBLE:

    inicio = time.perf_counter()

    try:
        respuesta = requests.get(
            API_URL,
            timeout=config["timeout"]
        )

        status_api = (
            respuesta.status_code
        )

        respuesta.raise_for_status()

        datos_api = (
            respuesta.json()
        )

        print(
            "Registros API:",
            len(
                datos_api
            )
        )

    except requests.RequestException as error:
        print(
            type(error).__name__,
            "->",
            error
        )

    tiempo_api = (
        time.perf_counter()
        - inicio
    )

## 5.1 Fallback local

In [ ]:
if datos_api is None:

    datos_api = [
        {
            "id": 1,
            "name": "Empresa Alfa",
            "email": "alfa@example.com",
            "address": {
                "city": "Barcelona"
            }
        },
        {
            "id": 2,
            "name": "Empresa Beta",
            "email": "beta@example.com",
            "address": {
                "city": "Madrid"
            }
        },
        {
            "id": 3,
            "name": "Empresa Gamma",
            "email": "gamma@example.com",
            "address": {
                "city": "Valencia"
            }
        }
    ]

    status_api = 200
    tiempo_api = 0

    print(
        "Usando fallback local."
    )

## 6. Normalizar respuesta API

In [ ]:
def normalizar_api(
    registros
):
    filas = []

    for item in registros:

        filas.append({
            "id":
                item.get("id"),

            "nombre":
                item.get(
                    "name"
                ),

            "email":
                item.get(
                    "email"
                ),

            "ciudad":
                item.get(
                    "address",
                    {}
                ).get(
                    "city"
                )
        })

    return pd.DataFrame(
        filas
    )


df_api = normalizar_api(
    datos_api
)

df_api.head()

## 6.1 Validar esquema API

In [ ]:
def validar_fuente_api(
    df
):
    requeridas = {
        "id",
        "nombre",
        "email",
        "ciudad"
    }

    faltantes = (
        requeridas
        - set(
            df.columns
        )
    )

    errores = []

    if faltantes:
        errores.append(
            f"Faltan columnas: {sorted(faltantes)}"
        )

    if df[
        "id"
    ].isna().any():
        errores.append(
            "Hay IDs ausentes."
        )

    return errores


validar_fuente_api(
    df_api
)

## 7. Segunda fuente: HTML

In [ ]:
html_local = '''
<html>
<body>
<table id="metricas">
    <thead>
        <tr>
            <th>id</th>
            <th>facturacion</th>
            <th>incidencias</th>
        </tr>
    </thead>
    <tbody>
        <tr><td>1</td><td>125000</td><td>4</td></tr>
        <tr><td>2</td><td>98000</td><td>12</td></tr>
        <tr><td>3</td><td>143500</td><td>3</td></tr>
    </tbody>
</table>
</body>
</html>
'''

print(
    html_local
)

## 7.1 Extraer tabla HTML

In [ ]:
if BS4_DISPONIBLE:

    soup = BeautifulSoup(
        html_local,
        "html.parser"
    )

    tabla = soup.select_one(
        "#metricas"
    )

    cabeceras = [
        th.get_text(
            strip=True
        )
        for th
        in tabla.select(
            "thead th"
        )
    ]

    filas = []

    for tr in tabla.select(
        "tbody tr"
    ):

        filas.append([
            td.get_text(
                strip=True
            )
            for td
            in tr.select(
                "td"
            )
        ])

    df_web = pd.DataFrame(
        filas,
        columns=cabeceras
    )

    df_web

## 7.2 Convertir tipos

In [ ]:
if BS4_DISPONIBLE:

    df_web["id"] = (
        df_web["id"]
        .astype(int)
    )

    df_web["facturacion"] = (
        df_web["facturacion"]
        .astype(float)
    )

    df_web["incidencias"] = (
        df_web["incidencias"]
        .astype(int)
    )

    df_web.dtypes

## 8. Provocar inconsistencia entre fuentes

In [ ]:
if BS4_DISPONIBLE:

    df_web_problematico = (
        df_web.copy()
    )

    df_web_problematico.loc[
        2,
        "id"
    ] = 999

    df_web_problematico

## 8.1 Detectar IDs sin correspondencia

In [ ]:
if BS4_DISPONIBLE:

    ids_api = set(
        df_api["id"]
    )

    ids_web = set(
        df_web_problematico[
            "id"
        ]
    )

    print(
        "Solo API:",
        ids_api - ids_web
    )

    print(
        "Solo Web:",
        ids_web - ids_api
    )

### Interpretación

Antes de hacer un merge debemos comprobar que las claves de integración son compatibles.

## 8.2 Corregir fuente web

In [ ]:
if BS4_DISPONIBLE:

    df_web_limpio = (
        df_web.copy()
    )

    print(
        set(
            df_api["id"]
        )
        - set(
            df_web_limpio["id"]
        )
    )

## 9. Integrar fuentes

In [ ]:
if BS4_DISPONIBLE:

    df_integrado = df_api.merge(
        df_web_limpio,
        on="id",
        how="left",
        validate="one_to_one"
    )

    df_integrado

## 9.1 Validar integración

In [ ]:
if BS4_DISPONIBLE:

    print(
        "Filas API:",
        len(
            df_api
        )
    )

    print(
        "Filas integradas:",
        len(
            df_integrado
        )
    )

    assert (
        len(
            df_api
        )
        == len(
            df_integrado
        )
    )

    print(
        "Merge validado."
    )

## 10. Crear métricas derivadas

In [ ]:
if BS4_DISPONIBLE:

    df_integrado[
        "nivel_facturacion"
    ] = pd.cut(
        df_integrado[
            "facturacion"
        ],
        bins=[
            -1,
            100000,
            130000,
            float("inf")
        ],
        labels=[
            "Baja",
            "Media",
            "Alta"
        ]
    )

    df_integrado[
        "requiere_revision"
    ] = (
        df_integrado[
            "incidencias"
        ] > 10
    )

    df_integrado

## 10.1 Visualizar facturación

In [ ]:
if BS4_DISPONIBLE:

    plt.figure(
        figsize=(8, 5)
    )

    plt.bar(
        df_integrado[
            "nombre"
        ],
        df_integrado[
            "facturacion"
        ]
    )

    plt.ylabel(
        "Facturación"
    )

    plt.title(
        "Facturación integrada por entidad"
    )

    plt.xticks(
        rotation=20
    )

    plt.tight_layout()
    plt.show()

## 11. Guardar dataset procesado

In [ ]:
if BS4_DISPONIBLE:

    ruta_csv = (
        raiz
        / config[
            "csv_output"
        ]
    )

    ruta_csv.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    df_integrado.to_csv(
        ruta_csv,
        index=False
    )

    print(
        ruta_csv.exists()
    )

## 12. Generar Excel final

In [ ]:
if (
    BS4_DISPONIBLE
    and OPENPYXL_DISPONIBLE
):

    ruta_excel = (
        raiz
        / config[
            "excel_output"
        ]
    )

    wb = Workbook()

    ws_datos = wb.active
    ws_datos.title = "Datos"

    ws_resumen = wb.create_sheet(
        "Resumen"
    )

    # Datos
    ws_datos.append(
        list(
            df_integrado.columns
        )
    )

    for fila in df_integrado.itertuples(
        index=False,
        name=None
    ):
        ws_datos.append(
            list(
                fila
            )
        )

    # Cabecera
    for celda in ws_datos[1]:
        celda.font = Font(
            bold=True,
            color="FFFFFF"
        )

        celda.fill = PatternFill(
            fill_type="solid",
            fgColor="1F4E78"
        )

        celda.alignment = Alignment(
            horizontal="center"
        )

    ws_datos.freeze_panes = "A2"

    # Resumen
    ws_resumen.append([
        "Entidad",
        "Facturacion",
        "Incidencias"
    ])

    for fila in df_integrado[
        [
            "nombre",
            "facturacion",
            "incidencias"
        ]
    ].itertuples(
        index=False,
        name=None
    ):
        ws_resumen.append(
            list(
                fila
            )
        )

    for celda in ws_resumen[1]:
        celda.font = Font(
            bold=True
        )

    # Gráfico
    chart = BarChart()

    chart.title = (
        "Facturación por entidad"
    )

    datos_chart = Reference(
        ws_resumen,
        min_col=2,
        min_row=1,
        max_row=ws_resumen.max_row
    )

    categorias = Reference(
        ws_resumen,
        min_col=1,
        min_row=2,
        max_row=ws_resumen.max_row
    )

    chart.add_data(
        datos_chart,
        titles_from_data=True
    )

    chart.set_categories(
        categorias
    )

    ws_resumen.add_chart(
        chart,
        "E2"
    )

    for ws in [
        ws_datos,
        ws_resumen
    ]:
        for columna in ws.columns:
            letra = get_column_letter(
                columna[0].column
            )

            maximo = max(
                len(
                    str(
                        celda.value
                    )
                )
                if celda.value is not None
                else 0
                for celda in columna
            )

            ws.column_dimensions[
                letra
            ].width = min(
                maximo + 2,
                30
            )

    wb.save(
        ruta_excel
    )

    print(
        ruta_excel.exists()
    )

## 12.1 Reabrir y validar Excel

In [ ]:
if (
    BS4_DISPONIBLE
    and OPENPYXL_DISPONIBLE
):

    wb_final = load_workbook(
        ruta_excel
    )

    print(
        wb_final.sheetnames
    )

    print(
        "Gráficos:",
        len(
            wb_final[
                "Resumen"
            ]._charts
        )
    )

    assert {
        "Datos",
        "Resumen"
    }.issubset(
        set(
            wb_final.sheetnames
        )
    )

## 13. Auditoría del pipeline

In [ ]:
auditoria = pd.DataFrame([
    {
        "fase":
            "API",

        "estado":
            "ok"
            if datos_api is not None
            else "error",

        "registros":
            len(
                df_api
            ),

        "detalle":
            f"status={status_api}"
    },

    {
        "fase":
            "Scraping",

        "estado":
            "ok"
            if BS4_DISPONIBLE
            else "omitido",

        "registros":
            len(
                df_web
            )
            if BS4_DISPONIBLE
            else 0,

        "detalle":
            "HTML local"
    },

    {
        "fase":
            "Integracion",

        "estado":
            "ok"
            if BS4_DISPONIBLE
            else "omitido",

        "registros":
            len(
                df_integrado
            )
            if BS4_DISPONIBLE
            else 0,

        "detalle":
            "merge one_to_one"
    }
])

auditoria

## 13.1 Guardar auditoría

In [ ]:
ruta_auditoria = (
    raiz
    / "logs"
    / "auditoria.csv"
)

auditoria.to_csv(
    ruta_auditoria,
    index=False
)

print(
    ruta_auditoria
)

## 14. Crear `.gitignore`

In [ ]:
gitignore = '''
.venv/
.env
__pycache__/
*.pyc
.ipynb_checkpoints/
data/processed/*
outputs/*
logs/*
'''

(
    raiz
    / ".gitignore"
).write_text(
    textwrap.dedent(
        gitignore
    ).strip()
    + "\n",
    encoding="utf-8"
)

print(
    (raiz / ".gitignore").read_text(
        encoding="utf-8"
    )
)

## 15. Crear README

In [ ]:
readme = '''
# Proyecto Automatización de Datos y Servicios

## Flujo

API + scraping → validación → integración → Excel.

## Instalación

```bash
python -m venv .venv
python -m pip install requests beautifulsoup4 pandas openpyxl python-dotenv
```

## Configuración

Copiar `.env.example` a `.env`.

## Ejecución

Ejecutar el notebook o modularizar el pipeline en `src/`.

## Salidas

- CSV integrado
- Excel final
- auditoría
'''

(
    raiz
    / "README.md"
).write_text(
    textwrap.dedent(
        readme
    ).strip()
    + "\n",
    encoding="utf-8"
)

print(
    (raiz / "README.md").read_text(
        encoding="utf-8"
    )
)

## 16. Git/GitHub: flujo conceptual

```bash
git init
git status
git add .
git commit -m "Initial automation project"

git branch -M main
git remote add origin <URL_REPOSITORIO>
git push -u origin main
```

Después:

```bash
git checkout -b feature/api-source
git add .
git commit -m "Add API source"
git push
```

## 17. Validar estructura final

In [ ]:
obligatorios = [
    raiz / "config" / "config.json",
    raiz / ".env.example",
    raiz / ".gitignore",
    raiz / "README.md",
    raiz / "logs" / "auditoria.csv"
]

estado = pd.DataFrame({
    "ruta": [
        str(
            ruta.relative_to(
                raiz
            )
        )
        for ruta
        in obligatorios
    ],

    "existe": [
        ruta.exists()
        for ruta
        in obligatorios
    ]
})

estado

## 18. Arquitectura final

In [ ]:
pd.DataFrame({
    "componente": [
        "API",
        "Scraping",
        "Validación",
        "Integración",
        "Config JSON",
        ".env",
        "CSV",
        "Excel",
        "Auditoría",
        "Git/GitHub"
    ],
    "responsabilidad": [
        "Fuente 1",
        "Fuente 2",
        "Calidad",
        "Unificación",
        "Parámetros",
        "Secretos/entorno",
        "Persistencia",
        "Reporting",
        "Trazabilidad",
        "Versionado"
    ]
})

## 19. Mini-proyecto final

Construye un proyecto con:

1. una API real o simulada;
2. una fuente HTML;
3. validación de ambas;
4. normalización;
5. integración;
6. métricas derivadas;
7. configuración JSON;
8. `.env`;
9. CSV final;
10. Excel final;
11. gráfico;
12. auditoría;
13. `.gitignore`;
14. README;
15. estructura reusable;
16. flujo Git/GitHub documentado.

## 20. Ejercicios finales

### Ejercicio 1
Crea estructura profesional.

### Ejercicio 2
Crea `config.json`.

### Ejercicio 3
Crea `.env.example`.

### Ejercicio 4
Consume una API.

### Ejercicio 5
Añade fallback local.

### Ejercicio 6
Normaliza JSON.

### Ejercicio 7
Extrae tabla HTML.

### Ejercicio 8
Provoca claves incompatibles.

### Ejercicio 9
Detecta IDs sin correspondencia.

### Ejercicio 10
Integra con `merge`.

### Ejercicio 11
Valida `one_to_one`.

### Ejercicio 12
Calcula métricas derivadas.

### Ejercicio 13
Exporta CSV.

### Ejercicio 14
Genera Excel.

### Ejercicio 15
Añade gráfico.

### Ejercicio 16
Genera auditoría.

### Ejercicio 17
Crea `.gitignore` y README.

### Ejercicio 18
Reto: construye el proyecto completo de automatización de datos y servicios.

## 21. Preguntas de reflexión

1. ¿Por qué combinar varias fuentes?
2. ¿Qué problemas aparecen al integrar datos?
3. ¿Por qué validar claves?
4. ¿Qué aporta `merge(validate=...)`?
5. ¿Por qué separar configuración?
6. ¿Qué debe ir en `.env`?
7. ¿Qué debe ir en JSON?
8. ¿Qué aporta una auditoría?
9. ¿Por qué generar CSV además de Excel?
10. ¿Qué aporta Git?
11. ¿Qué aporta GitHub?
12. ¿Cómo convertirías este notebook en un script ejecutable?
13. ¿Cómo automatizarías su ejecución diaria?
14. ¿Qué parte podría conectarse después con n8n?

## 22. Cierre

Con este laboratorio cerramos la parte de automatización de datos y servicios.

La progresión ha sido:

```text
Excel
  ↓
reporting
  ↓
PDF/LaTeX
  ↓
generación masiva
  ↓
APIs
  ↓
scraping
  ↓
proyecto integrado
```

El siguiente bloque cambia de paradigma:

```text
n8n
 ↓
workflows
 ↓
triggers
 ↓
nodos
 ↓
integración low-code
```